In [2]:
import requests
import io
import pandas as pd
import matplotlib.pyplot as plt

In [11]:
def extraer_tablas_investigacion(nombre_estrella):
    """
    Extrae parámetros físicos de la estrella y planetas (incluyendo el semi-eje mayor).
    Maneja apóstrofes en los nombres y estandariza los datos nulos.
    """
    url_base = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync"
    
    # Sanitizamos el string para evitar inyección SQL
    nombre_estrella_sql = nombre_estrella.replace("'", "''")
    
    # Añadimos pl_orbsmax a la consulta
    query = f"""
    SELECT 
        hostname AS estrella,
        st_spectype AS tipo_espectral,
        st_teff AS temp_estrella_k,
        st_rad AS radio_estrella_sol,
        st_mass AS masa_estrella_sol,
        st_teff_reflink AS ref_estrella,
        pl_name AS nombre_planeta,
        pl_orbper AS periodo_dias,
        pl_orbsmax AS semi_eje_ua,
        pl_rade AS radio_tierra,
        pl_bmasse AS masa_tierra,
        pl_bmassj AS masa_jupiter,
        pl_msinie AS mpsini_tierra,
        pl_msinij AS mpsini_jupiter,
        discoverymethod AS metodo_descubrimiento,
        pl_orbper_reflink AS ref_planeta
    FROM pscomppars
    WHERE hostname = '{nombre_estrella_sql}'
    """
    
    parametros = {
        "query": query,
        "format": "csv"
    }
    
    print(f"Buscando el sistema {nombre_estrella} en la base de datos...\n")
    respuesta = requests.get(url_base, params=parametros)
    
    if respuesta.status_code == 200:
        datos_csv = io.StringIO(respuesta.text)
        df_completo = pd.read_csv(datos_csv)
        
        if df_completo.empty:
            print(f"No se encontró información para '{nombre_estrella}'.")
            return None, None
            
        # 1. Tabla de la Estrella
        columnas_estrella = ['estrella', 'tipo_espectral', 'temp_estrella_k', 'radio_estrella_sol', 'masa_estrella_sol', 'ref_estrella']
        df_estrella = df_completo[columnas_estrella].drop_duplicates().reset_index(drop=True)
        df_estrella.columns = ['Estrella', 'Tipo_Espectral', 'Temp_K', 'Radio_Sol', 'Masa_Sol', 'Referencia_Estrella']
        
        # 2. Tabla de los Planetas (Ahora incluye semi_eje_ua)
        columnas_planetas = ['nombre_planeta', 'periodo_dias', 'semi_eje_ua', 'radio_tierra', 
                             'masa_tierra', 'masa_jupiter', 'mpsini_tierra', 'mpsini_jupiter', 
                             'metodo_descubrimiento', 'ref_planeta']
        df_planetas = df_completo[columnas_planetas].reset_index(drop=True)
        df_planetas.columns = ['Planeta', 'Periodo(Dias)', 'Semi_Eje(UA)', 'Radio(Tierra)', 
                               'Masa(Tierra)', 'Masa(Jupiter)', 'Mp_sin_i(Tierra)', 'Mp_sin_i(Jupiter)', 
                               'Metodo', 'Referencia_Planeta']
        
        return df_estrella, df_planetas
    else:
        print(f"Error HTTP {respuesta.status_code}: {respuesta.text}")
        return None, None

In [12]:
if __name__ == "__main__":
    
    estrella_objetivo = "Barnard's star" 
    
    df_star, df_planets = extraer_tablas_investigacion(estrella_objetivo)
    
    if df_star is not None and df_planets is not None:
        
        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', 1000)
        
        # ---------------------------------------------------------
        # PARTE 1: MOSTRAR RESULTADOS EN TABLA ASCII
        # ---------------------------------------------------------
        print("="*100)
        print(" " * 30 + "PARÁMETROS DE LA ESTRELLA ANFITRIONA")
        print("="*100)
        print(df_star.to_string(index=False, justify='center', na_rep='--'))
        print("\n")
        
        print("="*160)
        print(" " * 65 + "PARÁMETROS DE LOS EXOPLANETAS")
        print("="*160)
        print(df_planets.to_string(index=False, justify='center', na_rep='--'))
        print("="*160)
        print("\n")
        
        # ---------------------------------------------------------
        # PARTE 2: GUARDAR EN FORMATO OVERLEAF (.TEX)
        # ---------------------------------------------------------
        # Limpiamos los guiones bajos para que LaTeX no lance errores matemáticos en los títulos
        df_star.columns = [col.replace('_', ' ') for col in df_star.columns]
        df_planets.columns = [col.replace('_', ' ') for col in df_planets.columns]
        nombre_limpio = estrella_objetivo.replace("'", "").replace(" ", "_")

        # Nombres de archivos completamente dinámicos
        archivo_estrella = f"tabla_estrella_{nombre_limpio}.tex"
        archivo_planetas = f"tabla_planetas_{nombre_limpio}.tex"
        
        try:
            # Exportación moderna para Pandas 2.0+
            latex_star = df_star.style.format(na_rep='--').to_latex()
            latex_planets = df_planets.style.format(na_rep='--').to_latex()
        except AttributeError:
            # Compatibilidad para Pandas 1.x
            latex_star = df_star.to_latex(index=False, na_rep='--')
            latex_planets = df_planets.to_latex(index=False, na_rep='--')
            
        with open(archivo_estrella, 'w', encoding='utf-8') as f:
            f.write("% Generado automáticamente para Overleaf\n")
            f.write(latex_star)
            
        with open(archivo_planetas, 'w', encoding='utf-8') as f:
            f.write("% Generado automáticamente para Overleaf\n")
            f.write(latex_planets)

Buscando el sistema Barnard's star en la base de datos...

                              PARÁMETROS DE LA ESTRELLA ANFITRIONA
   Estrella    Tipo_Espectral  Temp_K  Radio_Sol  Masa_Sol                                                                           Referencia_Estrella                                                                          
Barnard's star    M3.5-4 V     3195.0   0.185      0.162   <a refstr=GONZALEZ_HERNANDEZ_ET_AL_2024 href=https://ui.adsabs.harvard.edu/abs/2024A&A...690A..79G/abstract target=ref>Gonz&aacute;lez Hern&aacute;ndez et al. 2024</a>


                                                                 PARÁMETROS DE LOS EXOPLANETAS
 Planeta   Periodo(Dias)  Semi_Eje(UA)  Radio(Tierra)  Masa(Tierra)  Masa(Jupiter)  Mp_sin_i(Tierra)  Mp_sin_i(Jupiter)      Metodo                                                             Referencia_Planeta                                                        
Barnard d     2.3402         0.0188        0.694        